# 104 職缺匯入 Qdrant

透過 `import_104_jobs_to_qdrant.py` 載入 Kaggle 的 104 職缺資料集，產生 embedding vector，匯入 Qdrant，並進行語意搜尋。



## 設定

In [ ]:
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchValue, PointStruct

from import_104_jobs_to_qdrant import (
    DEFAULT_COLLECTION_NAME,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_KAGGLE_DATASET,
    DEFAULT_QDRANT_URL,
    ensure_collection,
    get_embedding_model,
    get_vector_size,
    prepare_jobs,
)

INPUT_PATH = None  # 例如：Path("/path/to/jobs.csv")
DATASET_NAME = DEFAULT_KAGGLE_DATASET
QDRANT_URL = DEFAULT_QDRANT_URL
COLLECTION_NAME = DEFAULT_COLLECTION_NAME
EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL


## 載入資料

In [ ]:
prepared_jobs = prepare_jobs(INPUT_PATH, DATASET_NAME)

print(f"載入 {len(prepared_jobs)} 筆職缺")
print("\n--- 第一筆 embedding text ---")
print(prepared_jobs[0].text)


## 產生 Embedding Vector

In [ ]:
model = get_embedding_model(EMBEDDING_MODEL)
vectors = list(model.embed([job.text for job in prepared_jobs]))
vector_size = len(vectors[0]) if vectors else get_vector_size(EMBEDDING_MODEL)

print(f"vector 維度: {vector_size}")
print(f"共產生 {len(vectors)} 筆 vector")


## 建立 Collection 並匯入 Qdrant

In [ ]:
client = QdrantClient(url=QDRANT_URL)
ensure_collection(client, COLLECTION_NAME, vector_size)

points = [
    PointStruct(id=job.id, vector=vector.tolist(), payload=job.payload)
    for job, vector in zip(prepared_jobs, vectors, strict=True)
]
client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)

print(f"匯入完成，共 {len(points)} 筆")


## 驗證匯入

In [ ]:
count_result = client.count(collection_name=COLLECTION_NAME, exact=True)
sample_points = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[prepared_jobs[0].id],
    with_payload=True,
    with_vectors=False,
)
sample = sample_points[0].payload if sample_points else {}

print(f"預期筆數: {len(prepared_jobs)}")
print(f"實際筆數: {count_result.count}")
print(f"抽樣職缺: {sample.get('title')} / {sample.get('work_city')}")


## 語意搜尋

In [ ]:
QUERY = "會計 全職 台北"
CITY_FILTER = None  # 設城市過濾，例如 "台北市" 或 None
LIMIT = 5

query_vector = list(model.query_embed(QUERY))[0].tolist()

query_filter = None
if CITY_FILTER:
    query_filter = Filter(
        must=[FieldCondition(key="work_city", match=MatchValue(value=CITY_FILTER))]
    )

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    query_filter=query_filter,
    with_payload=True,
    with_vectors=False,
    limit=LIMIT,
)

for point in results.points:
    print(f"[{point.score:.4f}] {point.payload.get('title')} | {point.payload.get('company_name')} | {point.payload.get('work_city')}")
